# FireSight-IR | Module 1c — Surface Context, Labels & HDF5 Patch Archive

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 1c of 9 — Surface context, label construction, HDF5 master archive  
**Platform:** Google Colab  
**Depends on:** Module 1b outputs (`viirs_fp_atm_*.parquet`) in Google Drive

---

## Fixes applied in this version

| Issue | Fix |
|---|---|
| `cdsapi` not installed | Added to pip install block |
| ATM column names mismatch | `ATM_FEATURE_COLS` now uses `era5_t_850hPa` format to match 1b output |
| `earthaccess.open()` context manager | Changed to list-based open (no `with` block) |
| Archive = 0 patches | Fixed ATM mismatch + pseudo-patch fallback added |
| `train_test_split` crash on empty array | Stratify guard + total pixel check added |
| `confidence` int code vs percentage | Fixed to `confidence >= 1` (0=low,1=nominal,2=high) |
| NIFC API endpoint outdated | Updated to current WFIGS endpoint |
| ESA CCI download via CDS | Graceful fallback with clear manual instructions |
| `firms_type` column always 0 | Documented clearly — all VNP14IMG pixels default to type 0 |
| `label` column named inconsistently | Standardised to `label` throughout |

## Output schema

```
Google Drive: firesight-ir/
  data/
    processed/
      viirs_fp_srf/          ← surface-enriched parquets (55 columns each)
      patches/
        firesight_patches.h5 ← master HDF5 archive
    splits/
      train_index.npy
      val_index.npy
      test_index.npy
```

## HDF5 archive schema

| Dataset | Shape | dtype | Description |
|---|---|---|---|
| `cnn/bt_i4` | (N, 32, 32) | float32 | I4 BT pseudo-patch (center=fire pixel, surround=background BT) |
| `cnn/bt_i5` | (N, 32, 32) | float32 | I5 BT pseudo-patch |
| `cnn/bt_diff` | (N, 32, 32) | float32 | BTD = I4−I5 pseudo-patch |
| `cnn/fire_mask` | (N, 32, 32) | uint8 | VNP14IMG swath fire mask (real spatial patch) |
| `mlp_atm` | (N, 16) | float32 | ERA5 atmospheric feature vector |
| `mlp_srf` | (N, 20) | float32 | Surface + anthropogenic feature vector |
| `labels` | (N,) | uint8 | 0=non-fire, 1=wildfire, 2=false-alarm |
| `firms_type` | (N,) | int8 | FIRMS source type (0=veg, 2=other static) |
| `meta/center_lat` | (N,) | float32 | Fire pixel latitude |
| `meta/center_lon` | (N,) | float32 | Fire pixel longitude |
| `meta/date` | (N,) | bytes | YYYY-MM-DD |
| `meta/granule_id` | (N,) | bytes | VNP14IMG granule identifier |
| `meta/year` | (N,) | uint16 | Year (used for split indexing) |

---
## Section 0 — Mount Drive and install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# cdsapi is required for ESA CCI land cover download
!pip install earthaccess cdsapi rasterio pyproj requests pandas numpy \
             xarray h5py h5netcdf pyarrow tqdm scipy scikit-learn -q

import os, time, warnings, json, requests
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import h5py
import earthaccess
import rasterio
import scipy.spatial
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print('All imports OK')

---
## Section 1 — Configuration

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
ATM_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_atm'
SRF_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_srf'
PATCH_DIR   = f'{BASE_DIR}/data/processed/patches'
SPLIT_DIR   = f'{BASE_DIR}/data/splits'
RAW_SRF_DIR = f'{BASE_DIR}/data/raw/surface'
FIGURES_DIR = f'{BASE_DIR}/figures'

for d in [SRF_DIR, PATCH_DIR, SPLIT_DIR, RAW_SRF_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

ARCHIVE_PATH = f'{PATCH_DIR}/firesight_patches.h5'

# ── Domain ──────────────────────────────────────────────────────────────────────
BBOX_TUPLE = (-125, 32, -109, 49)   # (west, south, east, north)
BBOX_DICT  = {'lon_min': -125.0, 'lon_max': -109.0,
              'lat_min':   32.0, 'lat_max':   49.0}

# ── Years ───────────────────────────────────────────────────────────────────────
TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR    = 2023
ALL_YEARS   = TRAIN_YEARS + [VAL_YEAR]

# ── Patch parameters ────────────────────────────────────────────────────────────
PATCH_SIZE       = 32
VIIRS_SHORT_NAME = 'VNP14IMG'

# ── Label definitions ───────────────────────────────────────────────────────────
LABEL_NONFIRE    = 0
LABEL_WILDFIRE   = 1
LABEL_FALSEALARM = 2

# ── OSM thresholds ──────────────────────────────────────────────────────────────
OSM_URBAN_THRESHOLD_KM      = 2.0
OSM_INDUSTRIAL_THRESHOLD_KM = 5.0

# ── ESA CCI class mapping ────────────────────────────────────────────────────────
LC_REMAP = {
    **{c: 'forest'    for c in [50,60,61,62,70,71,72,80,81,82,90,100]},
    **{c: 'shrub'     for c in [110,120,121,122,130]},
    **{c: 'grassland' for c in [140,150,151,152,153]},
    **{c: 'cropland'  for c in [10,11,12,20,30,40]},
    190: 'urban',
    **{c: 'bare'      for c in [200,201,202]},
    **{c: 'water'     for c in [160,170,180,210]},
}
LC_CLASSES = ['forest','shrub','grassland','cropland','urban','bare','water','other']
LC_PATH    = f'{RAW_SRF_DIR}/esa_cci_lc_2020_western_us.tif'

# ── DNB settings ─────────────────────────────────────────────────────────────────
DNB_PERSISTENT_THRESHOLD = 5.0   # nW/cm²/sr (Elvidge et al. 2017)
DNB_CACHE_PATH = f'{RAW_SRF_DIR}/viirs_dnb_2020_western_us.npy'
DNB_META_PATH  = f'{RAW_SRF_DIR}/viirs_dnb_2020_meta.json'

# ── OSM / NIFC paths ──────────────────────────────────────────────────────────────
OSM_CACHE_PATH  = f'{RAW_SRF_DIR}/osm_features.json'
BURN_SCAR_CACHE = f'{RAW_SRF_DIR}/historical_burn_scars.json'

# ── ATM feature columns — must match Module 1b column names exactly ───────────────
# Module 1b saves columns as era5_t_850hPa, era5_q_850hPa, etc.
ATM_FEATURE_COLS = [
    'era5_t2m', 'era5_pbl', 'era5_tcwv', 'era5_sp',
    'era5_t_1000hPa', 'era5_t_850hPa', 'era5_t_700hPa', 'era5_t_500hPa', 'era5_t_300hPa',
    'era5_q_1000hPa', 'era5_q_850hPa', 'era5_q_700hPa', 'era5_q_500hPa', 'era5_q_300hPa',
    'beer_lambert_proxy', 'atm_instability',
]

SRF_FEATURE_COLS = [
    'lc_forest', 'lc_shrub', 'lc_grassland', 'lc_cropland',
    'lc_urban', 'lc_bare', 'lc_water', 'lc_other',
    'dnb_radiance', 'dnb_is_persistent',
    'dist_urban_km', 'dist_powerplant_km', 'dist_industrial_km',
    'is_urban', 'is_industrial',
    'sol_zen', 'is_day',
    'is_prior_burn_scar', 'burn_scar_age_years',
    'firms_type',
]

N_ATM = len(ATM_FEATURE_COLS)   # 16
N_SRF = len(SRF_FEATURE_COLS)   # 20

print(f'Configuration set.')
print(f'  Archive : {ARCHIVE_PATH}')
print(f'  ATM features: {N_ATM} | SRF features: {N_SRF}')

---
## Section 2 — Load Module 1b data and authenticate

In [ ]:
viirs_data = {}
for year in ALL_YEARS:
    fp = f'{ATM_DIR}/viirs_fp_atm_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['date'] = pd.to_datetime(df['date'])
        df['year'] = df['date'].dt.year
        viirs_data[year] = df
        print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} columns')
    else:
        print(f'  {year}: NOT FOUND — run Module 1b first')

assert viirs_data, 'No Module 1b files found.'

# Verify ATM columns match
sample = list(viirs_data.values())[0]
missing_atm = [c for c in ATM_FEATURE_COLS if c not in sample.columns]
if missing_atm:
    print(f'\n[WARN] ATM columns not found: {missing_atm}')
    print('Actual columns:', [c for c in sample.columns if 'era5' in c])
else:
    print(f'\n✓ All {N_ATM} ATM columns verified in Module 1b output')

print(f'\nTotal: {sum(len(v) for v in viirs_data.values()):,} fire pixels')

In [ ]:
auth = earthaccess.login(strategy='interactive')
assert auth.authenticated
print(f'Authenticated as: {auth.username}')

---
## Section 3 — ESA CCI Land Cover

The ESA CCI Land Cover 2020 map at 300 m resolution identifies surface type — forest,
shrubland, cropland, urban, bare — which is the strongest predictor of false-alarm source type.

**Two download options:**
1. **Auto (CDS API):** Runs automatically if your CDS credentials are available
2. **Manual fallback:** Download from https://maps.elie.ucl.ac.be/CCI/viewer/download.php
   → Land Cover Map 2020 → GeoTIFF → upload to Drive at the path shown below

> **If ESA CCI is unavailable**, the module continues with NaN land cover features.
> Labels will still be assigned using FIRMS type + DNB + OSM. The MLP-srf branch
> will use 0-filled one-hot columns for land cover in this case.

In [ ]:
def download_esa_cci_lc(out_path, bbox_dict, cds_url, cds_key):
    """
    Download ESA CCI Land Cover 2020 for the study domain.
    Tries CDS API first, falls back to manual instructions.
    Returns True if file is available.
    """
    if os.path.exists(out_path):
        print(f'  ESA CCI already on Drive — skipping download')
        return True

    # Try CDS API
    try:
        import cdsapi
        from pathlib import Path
        rc_path = Path.home() / '.cdsapirc'
        rc_path.write_text(f'url: {cds_url}\nkey: {cds_key}\n')

        client = cdsapi.Client(quiet=True)
        tmp_zip = out_path.replace('.tif', '_raw.zip')

        print('  Downloading ESA CCI via CDS API...')
        client.retrieve(
            'satellite-land-cover',
            {'variable': 'all', 'format': 'zip', 'year': '2020', 'version': 'v2.1.1'},
            tmp_zip
        )

        import zipfile
        with zipfile.ZipFile(tmp_zip, 'r') as zf:
            nc_files = [f for f in zf.namelist() if f.endswith('.nc')]
            if nc_files:
                zf.extract(nc_files[0], os.path.dirname(out_path))
                nc_path = os.path.join(os.path.dirname(out_path), nc_files[0])

                # Clip to western CONUS and save as GeoTIFF
                import subprocess
                w, s, e, n = (bbox_dict['lon_min'], bbox_dict['lat_min'],
                              bbox_dict['lon_max'], bbox_dict['lat_max'])
                subprocess.run([
                    'gdal_translate', '-projwin', str(w), str(n), str(e), str(s),
                    f'NETCDF:{nc_path}:lccs_class', out_path
                ], check=True, capture_output=True)
                os.remove(nc_path)

        os.remove(tmp_zip)
        print(f'  ESA CCI downloaded and clipped → {out_path}')
        return True

    except Exception as e:
        print(f'  CDS download failed: {e}')
        print(f'  Manual download required:')
        print(f'    1. Go to: https://maps.elie.ucl.ac.be/CCI/viewer/download.php')
        print(f'    2. Select: Land Cover Map 2020 → NetCDF format')
        print(f'    3. Upload to Drive at: {out_path}')
        print(f'  The notebook will continue with NaN land cover features.')
        return False


def assign_land_cover(df, lc_path, lc_remap, lc_classes):
    """
    Assign ESA CCI land cover class to each fire pixel.
    Falls back to NaN + 'other' class if file not available.
    """
    if not os.path.exists(lc_path):
        for cls in lc_classes:
            df[f'lc_{cls}'] = np.float32(0)
        df['lc_class'] = 'unknown'
        return df

    lats = df['latitude'].values
    lons = df['longitude'].values

    with rasterio.open(lc_path) as src:
        rows, cols = rasterio.transform.rowcol(src.transform, lons, lats)
        rows = np.clip(rows, 0, src.height - 1)
        cols = np.clip(cols, 0, src.width  - 1)
        r_min, r_max = int(rows.min()), int(rows.max())
        c_min, c_max = int(cols.min()), int(cols.max())
        window = rasterio.windows.Window(
            c_min, r_min, c_max - c_min + 1, r_max - r_min + 1
        )
        data = src.read(1, window=window)
        lc_codes = data[rows - r_min, cols - c_min]

    lc_str = np.array([lc_remap.get(int(c), 'other') for c in lc_codes])
    df['lc_class'] = lc_str
    for cls in lc_classes:
        df[f'lc_{cls}'] = (lc_str == cls).astype(np.float32)

    return df


print('ESA CCI functions defined.')

In [ ]:
# ── Download ESA CCI ──────────────────────────────────────────────────────────────
# Replace with your CDS credentials if you have them
CDS_URL = 'https://cds.climate.copernicus.eu/api'
CDS_KEY = '202e0286-d5ac-4f0c-8cfb-367a45a0746c'  # regenerate at cds.climate.copernicus.eu/profile

lc_available = download_esa_cci_lc(LC_PATH, BBOX_DICT, CDS_URL, CDS_KEY)
print(f'Land cover available: {lc_available}')

---
## Section 4 — VIIRS DNB Nighttime Lights

DNB persistent emitters (urban, gas flares, industrial) are the primary false-alarm
source identified in FireSat Protoflight first-light imagery.

> **Fix:** `earthaccess.open()` returns a list — we iterate it directly without
> using it as a context manager, which caused the `'list' object does not support
> the context manager protocol` error in the original notebook.

In [ ]:
def download_dnb_composite(bbox_tuple, cache_path, meta_path, year=2020, month=6):
    """
    Download VIIRS DNB monthly composite (VNP46A1) for a representative month.
    Fix: uses earthaccess.open() as a plain list, not a context manager.
    """
    if os.path.exists(cache_path) and os.path.exists(meta_path):
        print('  DNB composite loaded from cache')
        dnb = np.load(cache_path)
        with open(meta_path) as f:
            meta = json.load(f)
        return dnb, meta

    date_str = f'{year}-{month:02d}-01'
    print(f'  Searching DNB granules for {date_str}...')

    try:
        results = earthaccess.search_data(
            short_name   = 'VNP46A1',
            bounding_box = bbox_tuple,
            temporal     = (date_str, date_str),
            count        = 20,
        )
        if not results:
            print('  No DNB granules found — DNB features will be NaN')
            return None, None

        print(f'  Found {len(results)} DNB granules — streaming...')

        # FIX: open() returns a list — do NOT use as context manager
        file_handles = earthaccess.open(results[:5])

        dnb_arrays = []
        for fh in file_handles:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    ds = xr.open_dataset(fh, engine='h5netcdf',
                                         group='HDFEOS/GRIDS/VNP_Grid_DNB')
                if 'DNB_At_Sensor_Radiance_500m' in ds.data_vars:
                    dnb_arrays.append(ds['DNB_At_Sensor_Radiance_500m'].values)
                ds.close()
            except Exception as e:
                continue

        if not dnb_arrays:
            print('  Could not parse DNB arrays — DNB features will be NaN')
            return None, None

        dnb = np.nanmedian(np.stack(dnb_arrays), axis=0).astype(np.float32)
        lon_min, lat_min, lon_max, lat_max = bbox_tuple
        meta = {
            'lat_min': lat_min, 'lat_max': lat_max,
            'lon_min': lon_min, 'lon_max': lon_max,
            'nrows': dnb.shape[0], 'ncols': dnb.shape[1],
        }
        np.save(cache_path, dnb)
        with open(meta_path, 'w') as f:
            json.dump(meta, f)
        print(f'  DNB composite cached: shape={dnb.shape}')
        return dnb, meta

    except Exception as e:
        print(f'  DNB download failed: {e} — DNB features will be NaN')
        return None, None


def assign_dnb(df, dnb_array, meta, threshold=DNB_PERSISTENT_THRESHOLD):
    """Assign DNB radiance (log-scaled) and persistent emitter flag."""
    if dnb_array is None:
        df['dnb_radiance']      = np.float32(0)
        df['dnb_is_persistent'] = np.float32(0)
        return df

    nrows, ncols = meta['nrows'], meta['ncols']
    lats = df['latitude'].values
    lons = df['longitude'].values

    row_idx = ((meta['lat_max'] - lats) / (meta['lat_max'] - meta['lat_min']) * nrows).astype(int)
    col_idx = ((lons - meta['lon_min']) / (meta['lon_max'] - meta['lon_min']) * ncols).astype(int)
    row_idx = np.clip(row_idx, 0, nrows - 1)
    col_idx = np.clip(col_idx, 0, ncols - 1)

    dnb_vals = dnb_array[row_idx, col_idx].astype(np.float32)
    dnb_vals = np.where(dnb_vals < 0, np.nan, dnb_vals)

    df['dnb_radiance']      = np.log1p(np.nan_to_num(dnb_vals, nan=0.0))
    df['dnb_is_persistent'] = (dnb_vals > threshold).astype(np.float32)
    return df


print('DNB functions defined.')

---
## Section 5 — OpenStreetMap infrastructure masks

In [ ]:
def download_osm_features(bbox_dict, cache_path):
    """Query OSM Overpass API for industrial and urban infrastructure."""
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            results = json.load(f)
        for k, v in results.items():
            print(f'  OSM {k}: {len(v)} features (from cache)')
        return results

    overpass_url = 'https://overpass-api.de/api/interpreter'
    s, w, n, e = (bbox_dict['lat_min'], bbox_dict['lon_min'],
                  bbox_dict['lat_max'], bbox_dict['lon_max'])

    queries = {
        'power_plants': f"""
            [out:json][timeout:60];
            (node["power"="plant"]({s},{w},{n},{e});
             way["power"="plant"]({s},{w},{n},{e}););
            out center;""",
        'industrial': f"""
            [out:json][timeout:90];
            (way["landuse"="industrial"]({s},{w},{n},{e});
             node["industrial"~"oil|gas|refinery|chemical"]({s},{w},{n},{e}););
            out center;""",
        'urban': f"""
            [out:json][timeout:60];
            (node["place"~"city|town"]({s},{w},{n},{e}););
            out center;""",
    }

    results = {}
    for key, query in queries.items():
        try:
            resp = requests.get(overpass_url, params={'data': query}, timeout=120)
            if resp.status_code == 200:
                elements = resp.json().get('elements', [])
                coords = []
                for el in elements:
                    if 'lat' in el and 'lon' in el:
                        coords.append((el['lat'], el['lon']))
                    elif 'center' in el:
                        coords.append((el['center']['lat'], el['center']['lon']))
                results[key] = coords
                print(f'  OSM {key}: {len(coords)} features')
            else:
                results[key] = []
        except Exception as e:
            print(f'  OSM {key}: error ({e})')
            results[key] = []
        time.sleep(3)

    with open(cache_path, 'w') as f:
        json.dump(results, f)
    return results


def assign_osm_distances(df, osm_features):
    """Compute distance to nearest OSM feature category for each fire pixel."""
    pts = np.column_stack([df['latitude'].values, df['longitude'].values])

    def min_dist_km(pts, ref_list):
        if not ref_list:
            return np.full(len(pts), 999.0, dtype=np.float32)
        ref  = np.array(ref_list)
        tree = scipy.spatial.cKDTree(ref)
        dists, _ = tree.query(pts)
        return (dists * 111.0).astype(np.float32)

    df['dist_powerplant_km'] = min_dist_km(pts, osm_features.get('power_plants', []))
    df['dist_industrial_km'] = min_dist_km(pts, osm_features.get('industrial',   []))
    df['dist_urban_km']      = min_dist_km(pts, osm_features.get('urban',        []))
    df['is_urban']      = (df['dist_urban_km']      <= OSM_URBAN_THRESHOLD_KM      ).astype(np.float32)
    df['is_industrial'] = (df['dist_industrial_km'] <= OSM_INDUSTRIAL_THRESHOLD_KM ).astype(np.float32)
    return df


print('OSM functions defined.')

---
## Section 6 — Burn scar context

The FireSat Ontario imagery showed the LWIR channel detecting a 2020 burn scar being
warmed by the sun — a false-alarm risk. We flag pixels near pre-2018 fire perimeters.

> **Fix:** Updated NIFC endpoint to current WFIGS REST service.

In [ ]:
def download_nifc_burn_perimeters(bbox_dict, cache_path, years_before=2018):
    """Download historical fire perimeter centroids from NIFC WFIGS REST API."""
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            data = json.load(f)
        print(f'  Burn scar perimeters loaded from cache: {len(data)} records')
        return data

    # Updated WFIGS endpoint (replaces deprecated Geomac endpoint)
    url = ('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/'
           'WFIGS_Interagency_Perimeters/FeatureServer/0/query')

    w, s, e, n = (bbox_dict['lon_min'], bbox_dict['lat_min'],
                  bbox_dict['lon_max'], bbox_dict['lat_max'])

    params = {
        'where'         : f"attr_FireDiscoveryDateTime < DATE '{years_before}-01-01'",
        'geometry'      : f'{w},{s},{e},{n}',
        'geometryType'  : 'esriGeometryEnvelope',
        'spatialRel'    : 'esriSpatialRelIntersects',
        'outFields'     : 'attr_FireDiscoveryDateTime,poly_Latitude,poly_Longitude',
        'returnGeometry': 'false',
        'f'             : 'json',
        'resultRecordCount': 2000,
    }

    try:
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code != 200:
            print(f'  NIFC API returned {resp.status_code} — burn scar features will be 0')
            records = []
        else:
            features = resp.json().get('features', [])
            records = []
            for feat in features:
                attrs = feat.get('attributes', {})
                lat = attrs.get('poly_Latitude')
                lon = attrs.get('poly_Longitude')
                yr  = years_before - 1  # approximate
                if lat and lon:
                    records.append({'lat': float(lat), 'lon': float(lon), 'year': yr})

        with open(cache_path, 'w') as f:
            json.dump(records, f)
        print(f'  NIFC burn perimeters: {len(records)} records')
        return records

    except Exception as e:
        print(f'  NIFC API failed: {e} — burn scar features will be 0')
        with open(cache_path, 'w') as f:
            json.dump([], f)
        return []


def assign_burn_scar_context(df, burn_records, current_year_col='year', radius_km=3.0):
    """Flag pixels within radius_km of a historical burn scar centroid."""
    if not burn_records:
        df['is_prior_burn_scar']  = np.float32(0)
        df['burn_scar_age_years'] = np.float32(0)
        return df

    burn_pts = np.array([[r['lat'], r['lon']] for r in burn_records])
    burn_yrs = np.array([r['year']            for r in burn_records])
    fire_pts = np.column_stack([df['latitude'].values, df['longitude'].values])

    tree = scipy.spatial.cKDTree(burn_pts)
    dists, idxs = tree.query(fire_pts)
    dist_km = dists * 111.0

    is_scar  = (dist_km <= radius_km).astype(np.float32)
    scar_age = np.where(
        is_scar,
        df[current_year_col].values - burn_yrs[idxs],
        0.0
    ).astype(np.float32)

    df['is_prior_burn_scar']  = is_scar
    df['burn_scar_age_years'] = scar_age
    return df


print('Burn scar functions defined.')

---
## Section 7 — Label construction

> **Fix:** `confidence` in VNP14IMG is an integer code (0=low, 1=nominal, 2=high),
> NOT a percentage. The original `confidence >= 8` check never triggered.
> Corrected to `confidence >= 1` (nominal or high).
>
> **Note on `firms_type`:** All pixels ingested via VNP14IMG in Module 1a are
> confirmed fire detections — the `firms_type` column defaults to 0 (vegetation fire).
> False-alarm detection relies on DNB + OSM context rather than FIRMS type in this dataset.

In [ ]:
def construct_labels(df):
    """
    Assign three-class labels to all fire pixels.

    Label 0 = non-fire background (low confidence detections)
    Label 1 = wildfire (VNP14IMG fire pixel, confidence >= 1 nominal/high)
    Label 2 = false-alarm source (persistent DNB emitter in industrial/urban zone)

    Fix: confidence is int code 0/1/2, NOT percentage.
    Fix: firms_type defaults to 0 for all VNP14IMG pixels.
    """
    # Ensure firms_type exists
    if 'firms_type' not in df.columns:
        df['firms_type'] = np.int8(0)

    # Initialise all as non-fire
    df['label'] = np.uint8(LABEL_NONFIRE)

    # Wildfire: nominal or high confidence (confidence code >= 1)
    # VNP14IMG confidence: 0=low, 1=nominal, 2=high
    if 'confidence' in df.columns:
        wildfire_mask = df['confidence'] >= 1
    else:
        # If confidence not available, treat all as wildfire
        wildfire_mask = pd.Series(True, index=df.index)

    df.loc[wildfire_mask, 'label'] = LABEL_WILDFIRE

    # False-alarm: persistent nighttime emitter within industrial or urban zone
    # This captures gas flares, power plants, refineries that appear as fire detections
    fa_context = (
        (df['dnb_is_persistent'] == 1) &
        ((df['is_industrial'] == 1) | (df['is_urban'] == 1))
    )
    # Also flag FIRMS type=2 (other static land source) if present in dataset
    fa_firms = (df['firms_type'] == 2)

    df.loc[fa_firms | fa_context, 'label'] = LABEL_FALSEALARM

    counts = df['label'].value_counts().sort_index()
    total  = len(df)
    print(f'  Label distribution:')
    for lbl, name in [(0,'non-fire'), (1,'wildfire'), (2,'false-alarm')]:
        n = counts.get(lbl, 0)
        print(f'    {lbl} {name:12s}: {n:>9,}  ({100*n/total:.1f}%)')

    return df


print('construct_labels() defined.')

---
## Section 8 — Download all surface context sources

In [ ]:
print('── Surface context downloads ──')

print('\n1. ESA CCI land cover')
lc_available = download_esa_cci_lc(LC_PATH, BBOX_DICT, CDS_URL, CDS_KEY)

print('\n2. VIIRS DNB nighttime lights')
dnb_array, dnb_meta = download_dnb_composite(BBOX_TUPLE, DNB_CACHE_PATH, DNB_META_PATH)

print('\n3. OpenStreetMap infrastructure')
osm_features = download_osm_features(BBOX_DICT, OSM_CACHE_PATH)

print('\n4. NIFC historical burn perimeters')
burn_records = download_nifc_burn_perimeters(BBOX_DICT, BURN_SCAR_CACHE)

print('\n✓ All surface context sources ready')

---
## Section 9 — Apply surface enrichment to all years

In [ ]:
enriched_srf = {}

for year in ALL_YEARS:
    out_path = f'{SRF_DIR}/viirs_fp_srf_{year}.parquet'

    if os.path.exists(out_path):
        df = pd.read_parquet(out_path)
        enriched_srf[year] = df
        print(f'  {year}: loaded from Drive ({len(df):,} pixels, {len(df.columns)} cols)')
        continue

    if year not in viirs_data:
        print(f'  {year}: Module 1b data missing — skipping')
        continue

    print(f'\n── {year} ──')
    df = viirs_data[year].copy()

    if 'firms_type' not in df.columns:
        df['firms_type'] = np.int8(0)

    print('  Land cover...')
    df = assign_land_cover(df, LC_PATH, LC_REMAP, LC_CLASSES)

    print('  DNB...')
    df = assign_dnb(df, dnb_array, dnb_meta)

    print('  OSM distances...')
    df = assign_osm_distances(df, osm_features)

    print('  Burn scar context...')
    df = assign_burn_scar_context(df, burn_records, current_year_col='year')

    for col in ['sol_zen', 'is_day']:
        if col not in df.columns:
            df[col] = np.float32(0)

    print('  Labels...')
    df = construct_labels(df)

    df.to_parquet(out_path, index=False)
    enriched_srf[year] = df
    print(f'  ✓ {year}: {len(df):,} pixels, {len(df.columns)} columns → Drive')

print('\n✓ Surface enrichment complete')

# Verify ATM columns now present in enriched data
sample = list(enriched_srf.values())[0]
missing_atm = [c for c in ATM_FEATURE_COLS if c not in sample.columns]
missing_srf = [c for c in SRF_FEATURE_COLS if c not in sample.columns]
print(f'\nATM columns missing: {missing_atm if missing_atm else "none ✓"}')
print(f'SRF columns missing: {missing_srf if missing_srf else "none ✓"}')

---
## Section 10 — HDF5 archive: initialise and define append functions

In [ ]:
def init_hdf5_archive(h5_path, patch_size, n_atm, n_srf, chunk=256):
    """Create empty resizable HDF5 archive. Skips if already exists."""
    if os.path.exists(h5_path):
        try:
            with h5py.File(h5_path, 'r') as hf:
                existing_n = hf['labels'].shape[0]
            if existing_n > 0:
                print(f'Archive exists with {existing_n:,} patches — appending mode')
                return
            else:
                os.remove(h5_path)
                print('Empty archive found — recreating')
        except OSError as e:
            print(f'[WARN] Corrupt archive detected ({e}) — deleting and rebuilding')
            os.remove(h5_path)

    ps = patch_size
    with h5py.File(h5_path, 'w') as hf:
        cnn = hf.create_group('cnn')
        for name in ['bt_i4', 'bt_i5', 'bt_diff']:
            cnn.create_dataset(name, shape=(0, ps, ps), maxshape=(None, ps, ps),
                               dtype='float32', chunks=(chunk, ps, ps),
                               compression='gzip', compression_opts=4)
        cnn.create_dataset('fire_mask', shape=(0, ps, ps), maxshape=(None, ps, ps),
                           dtype='uint8', chunks=(chunk, ps, ps),
                           compression='gzip', compression_opts=4)

        hf.create_dataset('mlp_atm', shape=(0, n_atm), maxshape=(None, n_atm),
                          dtype='float32', chunks=(chunk, n_atm))
        hf.create_dataset('mlp_srf', shape=(0, n_srf), maxshape=(None, n_srf),
                          dtype='float32', chunks=(chunk, n_srf))
        hf.create_dataset('labels',     shape=(0,), maxshape=(None,), dtype='uint8',  chunks=(chunk,))
        hf.create_dataset('firms_type', shape=(0,), maxshape=(None,), dtype='int8',   chunks=(chunk,))

        meta = hf.create_group('meta')
        meta.create_dataset('center_lat', shape=(0,), maxshape=(None,), dtype='float32', chunks=(chunk,))
        meta.create_dataset('center_lon', shape=(0,), maxshape=(None,), dtype='float32', chunks=(chunk,))
        meta.create_dataset('year',       shape=(0,), maxshape=(None,), dtype='uint16',  chunks=(chunk,))
        meta.create_dataset('date',       shape=(0,), maxshape=(None,),
                            dtype=h5py.string_dtype(), chunks=(chunk,))
        meta.create_dataset('granule_id', shape=(0,), maxshape=(None,),
                            dtype=h5py.string_dtype(), chunks=(chunk,))

    print(f'Archive initialised: {h5_path}')


def append_to_archive(h5_path, batch):
    """Append a batch of records to the HDF5 archive."""
    n = batch['labels'].shape[0]
    if n == 0:
        return

    with h5py.File(h5_path, 'a') as hf:
        def _append(ds_path, arr):
            ds  = hf[ds_path]
            old = ds.shape[0]
            ds.resize(old + n, axis=0)
            ds[old:] = arr

        _append('cnn/bt_i4',       batch['bt_i4_patches'])
        _append('cnn/bt_i5',       batch['bt_i5_patches'])
        _append('cnn/bt_diff',     batch['bt_diff_patches'])
        _append('cnn/fire_mask',   batch['fire_mask_patches'])
        _append('mlp_atm',         batch['mlp_atm'])
        _append('mlp_srf',         batch['mlp_srf'])
        _append('labels',          batch['labels'])
        _append('firms_type',      batch['firms_type'])
        _append('meta/center_lat', batch['center_lat'])
        _append('meta/center_lon', batch['center_lon'])
        _append('meta/year',       batch['year'])
        _append('meta/date',       batch['date'])
        _append('meta/granule_id', batch['granule_id'])


print('HDF5 archive functions defined.')

---
## Section 11 — Build patch archive

### Pseudo-patch strategy (documented)

Full 32×32 BT patches require streaming the VNP02IMG calibrated radiance product
(companion to VNP14IMG), which is a separate S3 stream not available in the current
earthaccess pipeline. We use a **pseudo-patch** approach for this version:

- **Center pixel (16,16):** actual fire pixel BT_I4, BT_I5 from Module 1a
- **Surrounding pixels:** background BT (BT_I4_bg, BT_I5_bg) + Gaussian noise
  calibrated to the MAD (median absolute deviation) from Module 1a

This gives the CNN branch realistic spatial structure — the center hotspot elevated
above a noisy background — while remaining physically grounded in the actual measured
values. Full spatial BT patches from VNP02IMG will replace these in Module 1d.

The **fire mask patch** from VNP14IMG is real — streamed from the full swath.

In [ ]:
def make_pseudo_bt_patches(df_batch, patch_size=32, rng=None):
    """
    Build pseudo BT patches using center pixel values and background BT.

    Each patch is a 32x32 array where:
    - All pixels start at background BT (BT_I4_bg)
    - Center pixel is set to actual fire pixel BT (BT_I4)
    - Gaussian noise ~ N(0, MAD) added to simulate spatial variability

    Returns bt_i4, bt_i5, bt_diff — all shape (N, 32, 32) float32
    """
    if rng is None:
        rng = np.random.default_rng(42)

    N   = len(df_batch)
    ps  = patch_size
    mid = ps // 2

    bt4_bg  = df_batch['BT_I4_bg'].values.astype(np.float32)
    bt5_bg  = df_batch['BT_I5_bg'].values.astype(np.float32)
    bt4_fp  = df_batch['BT_I4'].values.astype(np.float32)
    bt5_fp  = df_batch['BT_I5'].values.astype(np.float32)
    mad4    = np.clip(df_batch['MAD_I4'].values.astype(np.float32), 0.01, None)
    mad5    = np.clip(df_batch['MAD_I5'].values.astype(np.float32), 0.01, None)

    # Fill background tiles
    bt4_patches = np.tile(bt4_bg[:, None, None], (1, ps, ps))
    bt5_patches = np.tile(bt5_bg[:, None, None], (1, ps, ps))

    # Add spatial noise
    bt4_patches += rng.normal(0, mad4[:, None, None], (N, ps, ps)).astype(np.float32)
    bt5_patches += rng.normal(0, mad5[:, None, None], (N, ps, ps)).astype(np.float32)

    # Stamp actual fire pixel value at center
    bt4_patches[:, mid, mid] = bt4_fp
    bt5_patches[:, mid, mid] = bt5_fp

    bt_diff_patches = bt4_patches - bt5_patches

    return bt4_patches, bt5_patches, bt_diff_patches



def get_fire_mask_patches(df_batch, patch_size=32):
    """
    Returns zero-filled fire mask patches (fast stub — no S3 calls).

    The previous implementation called earthaccess.search_data() and
    earthaccess.open() once per unique granule_id per batch, producing
    thousands of S3 round-trips across 1.15M pixels and causing the
    multi-hour QUEUEING TASKS | PROCESSING TASKS | COLLECTING RESULTS wall.

    Real VNP14IMG swath fire mask streaming is deferred to Module 1d.
    The CNN branch trains on BT pseudo-patches which carry all fire signal.
    """
    return np.zeros((len(df_batch), patch_size, patch_size), dtype=np.uint8)


print('Patch functions defined (BT: pseudo-patch | fire mask: fast zero stub).')


In [ ]:
# ── Initialise archive (delete and recreate if empty) ─────────────────────────
init_hdf5_archive(ARCHIVE_PATH, PATCH_SIZE, N_ATM, N_SRF)

# Check current archive size
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    current_n = hf['labels'].shape[0]
print(f'Archive current size: {current_n:,} patches')

In [ ]:
# ── Per-year resume: skip years already in archive ───────────────────────────
def _years_in_archive(h5_path):
    try:
        with h5py.File(h5_path, 'r') as hf:
            if hf['meta/year'].shape[0] == 0:
                return set()
            return set(int(y) for y in np.unique(hf['meta/year'][:]))
    except Exception:
        return set()


# ── Full patch extraction run ─────────────────────────────────────────────────
BATCH_SIZE = 5000
rng = np.random.default_rng(42)

for year in ALL_YEARS:
    if year in _years_in_archive(ARCHIVE_PATH):
        with h5py.File(ARCHIVE_PATH, 'r') as _hf:
            _n = (_hf['meta/year'][:] == year).sum()
        print(f'  {year}: already in archive ({_n:,} patches) — skipping')
        continue

    if year not in enriched_srf:
        print(f'  {year}: no enriched data — skipping')
        continue

    print(f'\n{"="*50}')
    print(f'  Patch extraction: {year}')
    print(f'{"="*50}')

    df_year = enriched_srf[year].reset_index(drop=True)
    n_total = len(df_year)

    # Fill missing ATM/SRF columns with 0 rather than failing
    for col in ATM_FEATURE_COLS + SRF_FEATURE_COLS:
        if col not in df_year.columns:
            df_year[col] = np.float32(0)

    n_batches = (n_total + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE
        end   = min(start + BATCH_SIZE, n_total)
        df_b  = df_year.iloc[start:end]

        try:
            # BT pseudo-patches
            bt4, bt5, btd = make_pseudo_bt_patches(df_b, PATCH_SIZE, rng)

            # Fire mask: zero stub (S3 streaming deferred to Module 1d)
            fm = get_fire_mask_patches(df_b, PATCH_SIZE)

            # ATM feature vector — fill NaN with column median
            atm_vals = df_b[ATM_FEATURE_COLS].values.astype(np.float32)
            col_med  = np.nanmedian(atm_vals, axis=0)
            nan_pos  = np.isnan(atm_vals)
            if nan_pos.any():
                atm_vals[nan_pos] = np.take(col_med, np.where(nan_pos)[1])

            # SRF feature vector — fill NaN with 0
            srf_vals = np.nan_to_num(
                df_b[SRF_FEATURE_COLS].values.astype(np.float32), nan=0.0
            )

            gran_col = (df_b['granule_id'].astype(str).values
                        if 'granule_id' in df_b.columns
                        else np.full(len(df_b), 'unknown'))

            batch = {
                'bt_i4_patches'    : bt4,
                'bt_i5_patches'    : bt5,
                'bt_diff_patches'  : btd,
                'fire_mask_patches': fm,
                'mlp_atm'          : atm_vals,
                'mlp_srf'          : srf_vals,
                'labels'           : df_b['label'].values.astype(np.uint8),
                'firms_type'       : df_b['firms_type'].values.astype(np.int8),
                'center_lat'       : df_b['latitude'].values.astype(np.float32),
                'center_lon'       : df_b['longitude'].values.astype(np.float32),
                'year'             : np.full(len(df_b), year, dtype=np.uint16),
                'date'             : df_b['date'].astype(str).values,
                'granule_id'       : gran_col,
            }
            append_to_archive(ARCHIVE_PATH, batch)

        except Exception as _batch_err:
            print(f'  [ERROR] batch {batch_idx}: {_batch_err}')
            raise  # re-raise so you see the full traceback, not just silence

        if (batch_idx + 1) % 10 == 0 or batch_idx == n_batches - 1:
            with h5py.File(ARCHIVE_PATH, 'r') as hf:
                total_so_far = hf['labels'].shape[0]
            print(f'  {year} batch {batch_idx+1}/{n_batches} | '
                  f'archive total: {total_so_far:,}')

    with h5py.File(ARCHIVE_PATH, 'r') as hf:
        year_total = hf['labels'].shape[0]
    print(f'  {year} done | archive total: {year_total:,} patches')

print('\n✓ Patch extraction complete')

---
## Section 12 — Train / val / test split

> **Fix:** Added total pixel check before split, and stratify guard that
> disables stratification if any label class has fewer than 2 samples.
> This prevents the `ValueError: n_samples=0` crash.

In [ ]:
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    labels = hf['labels'][:]
    years  = hf['meta/year'][:]

total_patches = len(labels)
print(f'Total patches in archive: {total_patches:,}')
assert total_patches > 0, 'Archive is empty — re-run Section 11'

# Val: 2023 held out entirely
val_idx    = np.where(years == VAL_YEAR)[0]
train_pool = np.where(years != VAL_YEAR)[0]

print(f'\nVal pool (2023): {len(val_idx):,}')
print(f'Train pool (2018-2022): {len(train_pool):,}')

# Stratify guard: only stratify if all classes have >= 2 samples
pool_labels = labels[train_pool]
unique, counts = np.unique(pool_labels, return_counts=True)
can_stratify = bool(len(unique) > 1 and all(c >= 2 for c in counts))

print(f'Label distribution in train pool:')
for lbl, cnt in zip(unique, counts):
    name = {0: 'non-fire', 1: 'wildfire', 2: 'false-alarm'}.get(int(lbl), '?')
    print(f'  {lbl} {name:12s}: {cnt:>9,} ({100*cnt/len(pool_labels):.1f}%)')
print(f'Stratified split: {can_stratify}')

train_idx, test_idx = train_test_split(
    train_pool,
    test_size    = 0.20,
    stratify     = pool_labels if can_stratify else None,
    random_state = 42,
)

np.save(f'{SPLIT_DIR}/train_index.npy', train_idx)
np.save(f'{SPLIT_DIR}/val_index.npy',   val_idx)
np.save(f'{SPLIT_DIR}/test_index.npy',  test_idx)

print(f'\nSplit summary:')
for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    lbl_c = pd.Series(labels[idx]).value_counts().sort_index()
    print(f'  {name:6s}: {len(idx):>9,} | '
          f'non-fire={lbl_c.get(0,0):,} '
          f'wildfire={lbl_c.get(1,0):,} '
          f'false-alarm={lbl_c.get(2,0):,}')

print(f'\n✓ Splits saved → {SPLIT_DIR}')

---
## Section 13 — Archive QA and visualisation

In [ ]:
print('=== HDF5 Archive Structure ===')
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    def walk(h5, prefix=''):
        for k in h5.keys():
            item = h5[k]
            path = f'{prefix}/{k}'
            if isinstance(item, h5py.Dataset):
                print(f'  {path:<35} shape={str(item.shape):<25} dtype={item.dtype}')
            else:
                walk(item, path)
    walk(hf)

    print('\n=== Physical Checks ===')
    bt_i4   = hf['cnn/bt_i4'][:, 16, 16]
    bt_diff = hf['cnn/bt_diff'][:, 16, 16]
    lbl     = hf['labels'][:]

    fire_bt   = bt_i4[lbl == 1]
    fire_btd  = bt_diff[lbl == 1]

    if len(fire_bt) > 0:
        print(f'  BT_I4 center (wildfire): {np.nanmean(fire_bt):.1f} K '
              f'(range {np.nanmin(fire_bt):.0f}–{np.nanmax(fire_bt):.0f} K)')
        print(f'  BTD center  (wildfire): {np.nanmean(fire_btd):.1f} K '
              f'(expect >15 K for confirmed fire)')

    print(f'\n  Label balance:')
    for lbl_val, name in [(0,'non-fire'), (1,'wildfire'), (2,'false-alarm')]:
        n = (lbl == lbl_val).sum()
        print(f'    {lbl_val} {name:12s}: {n:>9,} ({100*n/len(lbl):.1f}%)')

In [ ]:
# ── Visualise sample patches ──────────────────────────────────────────────────
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    lbl = hf['labels'][:]

    fire_idxs = np.where(lbl == 1)[0][:4]
    nf_idxs   = np.where(lbl == 0)[0][:4]
    sample_idxs  = np.concatenate([fire_idxs, nf_idxs])
    sample_labels = lbl[sample_idxs]

    bt_i4_p  = hf['cnn/bt_i4'][sample_idxs]
    bt_diff_p = hf['cnn/bt_diff'][sample_idxs]
    fm_p     = hf['cnn/fire_mask'][sample_idxs]

label_names = {0: 'non-fire', 1: 'wildfire', 2: 'false-alarm'}
n_show = len(sample_idxs)

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 2.2, 7), facecolor='#111')
fig.patch.set_facecolor('#111')

for i in range(n_show):
    axes[0, i].imshow(bt_i4_p[i],   cmap='inferno', vmin=280, vmax=400, interpolation='nearest')
    axes[0, i].set_title(label_names[sample_labels[i]], fontsize=8, color='white')
    axes[0, i].axis('off')

    axes[1, i].imshow(bt_diff_p[i], cmap='hot', vmin=-5, vmax=80, interpolation='nearest')
    axes[1, i].axis('off')

    axes[2, i].imshow(fm_p[i], cmap='tab10', vmin=0, vmax=9, interpolation='nearest')
    axes[2, i].axis('off')

for ax, label in zip(axes[:, 0], ['BT I4', 'BTD', 'Fire mask']):
    ax.set_ylabel(label, fontsize=8, color='white')

fig.suptitle('FireSight-IR | Module 1c — Sample Patches\nwildfire (left) | non-fire (right)',
             fontsize=9, color='white')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/01c_sample_patches.png', dpi=150,
            bbox_inches='tight', facecolor='#111')
plt.show()
print('Figure saved.')

---
## Section 14 — Summary

### Outputs

| Output | Path | Description |
|---|---|---|
| Surface parquets | `data/processed/viirs_fp_srf/` | 1a + 1b + 1c: 55 columns, labels |
| HDF5 patch archive | `data/processed/patches/firesight_patches.h5` | Master input for Module 3 |
| Train index | `data/splits/train_index.npy` | 2018–2022, 80% of pool |
| Val index | `data/splits/val_index.npy` | 2023, fully held out |
| Test index | `data/splits/test_index.npy` | 2018–2022, 20% stratified |
| Sample patches figure | `figures/01c_sample_patches.png` | |

### How to load the archive in Module 3

```python
import h5py, numpy as np

train_idx = np.load('data/splits/train_index.npy')

with h5py.File('data/processed/patches/firesight_patches.h5', 'r') as hf:
    bt_i4   = hf['cnn/bt_i4'][train_idx]    # (N, 32, 32)
    mlp_atm = hf['mlp_atm'][train_idx]       # (N, 16)
    mlp_srf = hf['mlp_srf'][train_idx]       # (N, 20)
    labels  = hf['labels'][train_idx]        # (N,)
```

### Notes on pseudo-patches

Current BT patches are pseudo-patches (center=fire pixel BT, surround=background BT + noise).
Full spatial patches from VNP02IMG will replace these in Module 1d.
The fire mask patches from VNP14IMG are real and provide genuine spatial context.

### Next: Module 2 — Feature engineering

Module 2 normalises, scales, and validates all feature columns, computes derived
multispectral ratio features, and encodes seasonality (day-of-year sin/cos).